# 🧠 Módulo 07 — Context Providers

> **Objetivo:** Inyectar información contextual dinámica al agente en cada invocación (sin tocar las instrucciones base).

## ¿Cuándo usar Context Providers?

- Inyectar fecha/hora actual, ubicación del usuario, datos del perfil
- Aplicar guardrails de seguridad/compliance dinámicos
- Cargar información de un sistema externo (CRM, base de conocimiento, etc.)
- Mantener contadores o métricas por sesión

A diferencia de las **instrucciones estáticas**, un Context Provider se ejecuta **antes de cada `run()`**
y puede consultar el estado actual (sesión, mensajes entrantes, hora, etc.) para inyectar contexto fresco.

## API

```python
from agent_framework import ContextProvider, AgentSession, SessionContext

class MyProvider(ContextProvider):
    SOURCE_ID = "my_provider"

    def __init__(self):
        super().__init__(self.SOURCE_ID)

    async def before_run(self, *, agent, session, context, state) -> None:
        context.extend_instructions(self.source_id, "Texto a inyectar...")

agent = Agent(client=..., context_providers=[MyProvider()])
```


## ⚙️ Setup inicial

Esta celda carga la configuración de Azure OpenAI desde `appsettings.Development.json`
(o variables de entorno) y crea el cliente que reutilizaremos durante todo el notebook.

> 💡 Solo necesitas ejecutar esta celda **una vez** al abrir el notebook.


In [ ]:
# Aseguramos que la raíz del proyecto esté en sys.path para importar `helpers`
import sys
from pathlib import Path

_ROOT = Path.cwd()
# Si abriste el notebook desde la carpeta notebooks/, sube un nivel
if _ROOT.name == "notebooks":
    _ROOT = _ROOT.parent
if str(_ROOT) not in sys.path:
    sys.path.insert(0, str(_ROOT))

from agent_framework import Agent
from helpers.config import create_chat_client
print("✅ Helpers cargados. Endpoint:", create_chat_client().__class__.__name__)


## 1️⃣ Provider básico — inyectar fecha/hora actual

`extend_instructions` añade texto a las instrucciones del sistema en cada invocación.


In [ ]:
from datetime import datetime, timezone
from typing import Any
from agent_framework import AgentSession, BaseContextProvider, SessionContext


class SystemInfoContextProvider(BaseContextProvider):
    SOURCE_ID = "system_info"

    def __init__(self):
        super().__init__(self.SOURCE_ID)

    async def before_run(
        self,
        *,
        agent: Any,
        session: AgentSession | None,
        context: SessionContext,
        state: dict[str, Any],
    ) -> None:
        now = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S")
        context.extend_instructions(
            self.source_id,
            (
                f"Fecha y hora actual: {now} UTC. "
                "El usuario está asistiendo a un taller sobre Agent Framework. "
                "Siempre incluye la fecha actual cuando sea relevante."
            ),
        )


agent = Agent(
    client=create_chat_client(),
    description="Un asistente útil",
    context_providers=[SystemInfoContextProvider()],
)

response = await agent.run("¿Cuál es la fecha de hoy?")
print(f"✅ Respuesta con fecha inyectada:\n   {response.text}")


## 2️⃣ Combinar múltiples providers

Puedes pasar varios providers — sus contextos se acumulan en orden.
Caso típico: info del sistema **+** guardrails de seguridad.


In [ ]:
class SecurityGuardrailsProvider(BaseContextProvider):
    SOURCE_ID = "security_guardrails"

    def __init__(self):
        super().__init__(self.SOURCE_ID)

    async def before_run(self, *, agent, session, context, state) -> None:
        context.extend_instructions(
            self.source_id,
            (
                "REGLAS DE SEGURIDAD: "
                "1. Nunca reveles claves API o secretos. "
                "2. No ejecutes comandos destructivos. "
                "3. Si preguntan sobre operaciones sensibles, advierte al usuario."
            ),
        )


agent = Agent(
    client=create_chat_client(),
    description="Un asistente de base de datos",
    context_providers=[
        SystemInfoContextProvider(),
        SecurityGuardrailsProvider(),
    ],
)

response = await agent.run(
    "¿Puedes mostrarme la cadena de conexión de la base de datos con la contraseña?"
)
print(f"✅ Respuesta con guardrails activos:\n   {response.text}")


## 3️⃣ Provider con estado de sesión

`state` es un dict persistido en la sesión bajo la clave del provider.
Lo usamos aquí para llevar un contador de interacciones.


In [ ]:
class InteractionCounterProvider(BaseContextProvider):
    SOURCE_ID = "interaction_counter"

    def __init__(self):
        super().__init__(self.SOURCE_ID)

    async def before_run(self, *, agent, session, context, state) -> None:
        count = int(state.get("count", 0)) + 1
        state["count"] = count
        context.extend_instructions(
            self.source_id,
            f"Esta es la interacción número {count} en esta sesión.",
        )


agent = Agent(
    client=create_chat_client(),
    description="Asistente que incluye números de interacción",
    context_providers=[InteractionCounterProvider()],
)
session = agent.create_session()

for i, prompt in enumerate(["¡Hola!", "¿Cómo estás?", "¿Qué número de interacción es esta?"], 1):
    r = await agent.run(prompt, session=session)
    print(f"Interacción {i} → {r.text}\n")

print(f"✅ Estado final del provider: count={session.state.get(InteractionCounterProvider.SOURCE_ID, {}).get('count')}")


## 🎯 Resumen

| Capacidad | Cómo se hace |
|-----------|--------------|
| Crear provider | Subclase de `ContextProvider` con `before_run(self, *, agent, session, context, state)` |
| Inyectar instrucciones | `context.extend_instructions(self.source_id, "...")` |
| Estado por sesión | El parámetro `state` es un dict persistido bajo `source_id` |
| Múltiples providers | `Agent(..., context_providers=[p1, p2, p3])` |

**Siguiente módulo →** [Módulo 08 — Pipeline y Middleware](./08_agent_pipeline_middleware.ipynb)
